# Instagram Sentiment Analysis - Example Notebook

This notebook demonstrates the complete workflow of the Instagram Sentiment Analyzer:
1. Data Collection
2. Text Preprocessing
3. Model Training
4. Sentiment Analysis
5. Visualization

## Setup and Imports

In [ ]:
# Install required packages if needed
# !pip install -r ../requirements.txt

# Import libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Add src to path
sys.path.append(str(Path().absolute().parent / "src"))

# Import our modules
from scraper import InstagramScraper
from preprocess import TextPreprocessor
from sentiment_model import SentimentClassifier
from visualize import SentimentVisualizer
from utils import ConfigManager, DataValidator

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete!")

## Load Configuration

In [ ]:
# Load configuration
config = ConfigManager.load_config("../config.json")

print("Configuration loaded:")
print(json.dumps(config, indent=2))

## 1. Data Collection

### Option A: Scrape Instagram Data

**Note**: You'll need Instagram credentials for scraping. For this example, we'll use sample data.

In [ ]:
# Initialize scraper
scraper = InstagramScraper(config)

# Example: Scrape posts from a profile
# Note: You need to login first
# scraper.login('your_username', 'your_password')
# posts_data = scraper.get_profile_posts('example_profile', max_posts=10)

# For this example, we'll create sample data
sample_posts = [
    {
        'shortcode': 'sample1',
        'caption': 'Amazing sunset today! Feeling so blessed and grateful for this beautiful view! 🌅 #sunset #blessed #nature',
        'hashtags': ['sunset', 'blessed', 'nature'],
        'mentions': [],
        'likes': 150,
        'comments_count': 25,
        'timestamp': '2024-01-15T18:30:00',
        'is_video': False,
        'location': None,
        'tagged_users': [],
        'comments': [
            {'text': 'Beautiful!', 'owner': 'user1', 'likes': 5, 'timestamp': '2024-01-15T18:35:00'},
            {'text': 'Love this view', 'owner': 'user2', 'likes': 3, 'timestamp': '2024-01-15T18:40:00'}
        ]
    },
    {
        'shortcode': 'sample2',
        'caption': 'Terrible service at this restaurant. Waited 2 hours for cold food. Very disappointed! 😠 #badservice #disappointed',
        'hashtags': ['badservice', 'disappointed'],
        'mentions': [],
        'likes': 45,
        'comments_count': 12,
        'timestamp': '2024-01-14T20:15:00',
        'is_video': False,
        'location': None,
        'tagged_users': [],
        'comments': [
            {'text': 'That sounds awful', 'owner': 'user3', 'likes': 2, 'timestamp': '2024-01-14T20:20:00'}
        ]
    },
    {
        'shortcode': 'sample3',
        'caption': 'Just an ordinary day at the office. Nothing exciting to report. #work #office #neutral',
        'hashtags': ['work', 'office', 'neutral'],
        'mentions': [],
        'likes': 78,
        'comments_count': 8,
        'timestamp': '2024-01-13T09:00:00',
        'is_video': False,
        'location': None,
        'tagged_users': [],
        'comments': []
    },
    {
        'shortcode': 'sample4',
        'caption': 'Absolutely love my new phone! The camera is incredible and battery life is amazing! 📱❤️ #newphone #tech #happy',
        'hashtags': ['newphone', 'tech', 'happy'],
        'mentions': [],
        'likes': 230,
        'comments_count': 35,
        'timestamp': '2024-01-12T14:20:00',
        'is_video': False,
        'location': None,
        'tagged_users': [],
        'comments': [
            {'text': 'Which phone did you get?', 'owner': 'user4', 'likes': 8, 'timestamp': '2024-01-12T14:25:00'}
        ]
    },
    {
        'shortcode': 'sample5',
        'caption': 'This movie was so boring and predictable. Waste of time and money. #badmovie #disappointed',
        'hashtags': ['badmovie', 'disappointed'],
        'mentions': [],
        'likes': 32,
        'comments_count': 6,
        'timestamp': '2024-01-11T21:45:00',
        'is_video': False,
        'location': None,
        'tagged_users': [],
        'comments': []
    }
]

# Convert to DataFrame
df = pd.DataFrame(sample_posts)

print(f"Sample data created with {len(df)} posts")
df.head()

### Option B: Load Existing Data

In [ ]:
# If you have existing data, you can load it like this:
# df = pd.read_csv('your_data.csv')
# or
# with open('your_data.json', 'r') as f:
#     data = json.load(f)
# df = pd.DataFrame(data)

print("Using sample data for demonstration")

## 2. Data Validation

In [ ]:
# Validate data structure
validation_results = DataValidator.validate_instagram_data(df)

print("Data Validation Results:")
print(f"Valid: {validation_results['is_valid']}")
print(f"Errors: {validation_results['errors']}")
print(f"Warnings: {validation_results['warnings']}")
print(f"Stats: {validation_results['stats']}")

## 3. Text Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor(config)

# Preprocess the data
processed_df = preprocessor.preprocess_dataframe(df, 'caption')

print("Preprocessing completed!")
print(f"Original text: {df['caption'].iloc[0]}")
print(f"Processed text: {processed_df['processed_text'].iloc[0]}")
print(f"Tokens: {processed_df['tokens'].iloc[0]}")

# Show preprocessing results
processed_df[['caption', 'cleaned_text', 'processed_text', 'token_count']].head()

## 4. Sentiment Model Training

In [ ]:
# Initialize classifier
classifier = SentimentClassifier(config)

# For this example, we'll create sample labels
# In production, you would need actual labeled data
np.random.seed(42)
texts = processed_df['processed_text'].tolist()

# Create realistic sample labels based on content
sample_labels = []
for text in texts:
    if any(word in text.lower() for word in ['amazing', 'love', 'beautiful', 'blessed', 'happy']):
        sample_labels.append('positive')
    elif any(word in text.lower() for word in ['terrible', 'bad', 'disappointed', 'boring']):
        sample_labels.append('negative')
    else:
        sample_labels.append('neutral')

print(f"Sample labels: {sample_labels}")

# Train the model
metrics = classifier.train_model(texts, sample_labels)

print(f"\nModel Training Results:")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Training samples: {metrics['train_samples']}")
print(f"Test samples: {metrics['test_samples']}")
print(f"Features: {metrics['features']}")

## 5. Sentiment Prediction

In [ ]:
# Make predictions on all texts
predictions = classifier.predict(texts)

# Add predictions to dataframe
analysis_df = processed_df.copy()
analysis_df['predicted_sentiment'] = [pred['predicted_label'] for pred in predictions]
analysis_df['confidence'] = [pred.get('confidence', 0) for pred in predictions]

# Display results
results_df = analysis_df[['caption', 'processed_text', 'predicted_sentiment', 'confidence', 'likes', 'comments_count']]
results_df.head()

## 6. Visualization

In [ ]:
# Initialize visualizer
visualizer = SentimentVisualizer(config)

# Create sentiment distribution pie chart
fig_pie = visualizer.sentiment_distribution_pie(analysis_df, 'predicted_sentiment')
fig_pie.show()

In [ ]:
# Create engagement vs sentiment correlation
# Calculate engagement rate
analysis_df['engagement_rate'] = (analysis_df['likes'] + analysis_df['comments_count']) / analysis_df['likes'].replace(0, 1)

fig_engagement = visualizer.engagement_sentiment_correlation(analysis_df, 'engagement_rate', 'predicted_sentiment')
fig_engagement.show()

In [ ]:
# Create word clouds for each sentiment
wordclouds = visualizer.word_cloud_by_sentiment(analysis_df, 'processed_text', 'predicted_sentiment')

for sentiment, fig in wordclouds.items():
    plt.figure(figsize=(10, 5))
    plt.imshow(fig, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'{sentiment.title()} Sentiment Word Cloud', fontsize=16)
    plt.show()

## 7. Summary Statistics

In [ ]:
# Get comprehensive summary
summary = visualizer.create_dashboard_summary(analysis_df)

print("=== ANALYSIS SUMMARY ===")
print(f"Total posts: {summary['total_posts']}")
print(f"Sentiment distribution: {summary['sentiment_distribution']}")
print(f"Average engagement: {summary['avg_engagement']:.2f}")
print(f"Posts per day: {summary['posting_frequency']['posts_per_day']:.1f}")
print(f"Most active day: {summary['posting_frequency']['most_active_day']}")
print(f"Peak posting hours: {summary['peak_hours']}")

if summary['top_hashtags']:
    print("\nTop hashtags:")
    for item in summary['top_hashtags'][:5]:
        print(f"  #{item['hashtag']}: {item['count']} posts")

## 8. Save Results

In [ ]:
# Create output directory
output_dir = Path("../data/notebook_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save analysis results
analysis_df.to_csv(output_dir / "sentiment_analysis_results.csv", index=False)
analysis_df.to_json(output_dir / "sentiment_analysis_results.json", orient='records', indent=2)

# Save model
classifier.save_model(str(output_dir / "sentiment_model.pkl"))

# Export visualizations
saved_files = visualizer.export_visualizations(analysis_df, str(output_dir / "visualizations"))

print(f"Results saved to: {output_dir}")
print(f"Visualization files: {saved_files}")

## 9. Feature Importance (if available)

In [ ]:
# Get feature importance for interpretable models
try:
    feature_importance = classifier.get_feature_importance(top_n=10)
    
    print("Feature Importance:")
    for sentiment, features in feature_importance.items():
        print(f"\n{sentiment.upper()}:")
        for word, score in features[:5]:
            print(f"  {word}: {score:.4f}")
except Exception as e:
    print(f"Feature importance not available: {e}")

## 10. Conclusion

This notebook demonstrated the complete Instagram Sentiment Analysis workflow:

1. **Data Collection**: We created sample Instagram post data with captions, hashtags, and engagement metrics
2. **Data Validation**: Validated the data structure and quality
3. **Text Preprocessing**: Cleaned and processed text using tokenization, stopword removal, and lemmatization
4. **Model Training**: Trained a sentiment classification model using TF-IDF embeddings
5. **Prediction**: Applied the model to predict sentiment for all posts
6. **Visualization**: Created comprehensive visualizations including pie charts, word clouds, and engagement analysis
7. **Analysis**: Generated summary statistics and insights

### Key Insights from Sample Data:
- The model successfully identified positive, negative, and neutral sentiments
- Word clouds revealed characteristic vocabulary for each sentiment category
- Engagement patterns varied by sentiment type

### Next Steps:
- Use real Instagram data with actual credentials
- Implement proper labeled data for supervised learning
- Try different models and embeddings for better accuracy
- Add more advanced features like topic modeling or emotion detection

To run the interactive dashboard, use:
```bash
streamlit run app/streamlit_app.py
```